# Prediction Accuracy Overview

Shows when predictions are correct vs incorrect (no attributions).

In [ ]:
import sys
from pathlib import Path

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir(): break
    _current = _current.parent
sys.path.insert(0, str(_current))
sys.path.insert(0, str(_current / 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm.auto import tqdm

from src.interpretability.config.domestic_declarations_config import CONFIG
from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM
from src.evaluation.evaluation import Evaluation

%matplotlib inline

In [ ]:
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(CONFIG.get_model_path()), dropout=0.0)
model.eval()
device = torch.device('cpu')
test_dataset = torch.load(str(CONFIG.get_test_data_path()), weights_only=False)
eval_helper = Evaluation(model=model, dataset=test_dataset, concept_name=CONFIG.concept_name,
                         growing_num_values=CONFIG.growing_num_values, all_cat=CONFIG.all_cat, all_num=CONFIG.all_num)

# Activity label mapping
concept_id = eval_helper.concept_name_id
inv_cats = eval_helper.inverted_suffix_categories[concept_id]

print(f"Cases: {len(eval_helper.cases)}, Suffix length: {model.seq_len_pred}")

In [ ]:
# Collect predictions - use suffix tensor for ground truth (like ProcessPerformanceMap)
results = []

for case_name, case in tqdm(eval_helper.cases.items(), desc="Evaluating"):
    for prefix_len, prefix, suffix in eval_helper._iterate_case(case):
        # Ground truth: first non-zero activity in suffix tensor
        suffix_cats = suffix[0]
        gt_idx = suffix_cats[concept_id][0, 0].item()
        if gt_idx == 0:  # Skip padding
            continue
        gt_activity = inv_cats.get(gt_idx, str(gt_idx))
        
        # Model prediction (first step only)
        with torch.no_grad():
            predictions, _, _, _ = model(prefix)
        
        cat_preds, num_preds = predictions
        activity_logits = cat_preds[f'{CONFIG.concept_name}_mean']
        if activity_logits.dim() == 3:
            activity_logits = activity_logits[0]  # First suffix step
        
        pred_idx = activity_logits.argmax(dim=-1).item()
        pred_activity = inv_cats.get(pred_idx, str(pred_idx))
        
        results.append({
            'prefix_len': prefix_len,
            'predicted': pred_activity,
            'ground_truth': gt_activity,
            'correct': pred_idx == gt_idx
        })

print(f"Total predictions: {len(results)}")
print(f"Overall accuracy: {sum(r['correct'] for r in results) / len(results):.1%}")

In [ ]:
# Accuracy by prefix length
prefix_lens = sorted(set(r['prefix_len'] for r in results))

accs, counts = [], []
for pl in prefix_lens:
    pl_results = [r for r in results if r['prefix_len'] == pl]
    accs.append(sum(r['correct'] for r in pl_results) / len(pl_results))
    counts.append(len(pl_results))

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(prefix_lens, accs, color='steelblue', alpha=0.7)
ax.set_xlabel('Prefix Length')
ax.set_ylabel('Accuracy')
ax.set_title('Next Activity Prediction Accuracy by Prefix Length')
ax.set_ylim(0, 1)
ax.axhline(y=sum(r['correct'] for r in results)/len(results), color='red', linestyle='--', label='Overall')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nAccuracy by prefix length:")
for pl, acc, cnt in zip(prefix_lens, accs, counts):
    print(f"  Prefix {pl}: {acc:.1%} (n={cnt})")

In [ ]:
# Most common errors
errors = [(r['ground_truth'], r['predicted']) for r in results if not r['correct']]
error_counts = Counter(errors)

print(f"Total errors: {len(errors)} ({len(errors)/len(results):.1%})")
print("\nTop 10 error patterns (ground_truth -> predicted):")
for (gt, pred), cnt in error_counts.most_common(10):
    print(f"  {gt} -> {pred}: {cnt}")

In [ ]:
# Per-activity accuracy
gt_to_preds = defaultdict(Counter)
for r in results:
    gt_to_preds[r['ground_truth']][r['predicted']] += 1

print("Per-activity accuracy:")
for gt in sorted(gt_to_preds.keys()):
    total = sum(gt_to_preds[gt].values())
    correct = gt_to_preds[gt].get(gt, 0)
    print(f"  {gt}: {correct}/{total} = {correct/total:.1%}")